# E2-03 · Comparación de dos o más grupos: pruebas t y ANOVA

**Estadística II · Ingeniería Industrial · Universidad Industrial de Santander**  
*Versión estudiante*

**Propósito de aprendizaje**

- Comparar las medias de dos grupos independientes eligiendo entre la t de Student, la t de Welch y Mann-Whitney según los supuestos.
- Plantear y resolver un ANOVA de un factor, verificar sus supuestos e identificar qué grupos difieren con Tukey.
- Tomar una decisión de compras, de calidad o de mantenimiento a partir de los resultados, distinguiendo significancia estadística de relevancia práctica.

**Duración sugerida:** 2 sesiones de 2 horas  
**Datos:** casos IC07 (precios de compra), IC05 (proveedores de acero) e IC03 (tiempos de reparación)

## Estructura

| Parte | Caso | Modalidad | Pregunta de negocio |
|---|---|---|---|
| 1 | IC07 · Precios de compra | guiada | ¿Comprar por contrato marco sale más barato que comprar en el mercado? |
| 2 | IC05 · Resistencia por proveedor | semi-guiada | ¿Los tres proveedores entregan acero con la misma resistencia media? |
| 3 | IC03 · Tiempos de reparación | autónoma | ¿El tiempo de reparación depende del tipo de equipo? |

En las celdas marcadas con `# Escriba su código aquí` usted completa el análisis. Las preguntas de interpretación se responden en las celdas de texto **✍️ Su respuesta**. Use α = 0,05.

# 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
sns.set_theme(style="whitegrid", palette=["#00ADC6", "#FF6F49", "#25A687", "#7B6FD0"])
ALFA = 0.05

import os
from pathlib import Path

def cargar_csv(nombre, carpeta="casos_industriales/csv"):
    """Busca el archivo en la carpeta actual o en la carpeta datos/ del repositorio.
    Si no lo encuentra y se está en Google Colab, abre el diálogo para subirlo."""
    candidatos = [Path(nombre)] + [Path(p) / "datos" / carpeta / nombre for p in [".", "..", "../..", "../../.."]]
    for ruta in candidatos:
        if ruta.exists():
            print(f"Leyendo {ruta}")
            return pd.read_csv(ruta)
    try:
        from google.colab import files  # solo existe en Colab
        print(f"No encontré '{nombre}'. Súbalo desde su computador:")
        subido = files.upload()
        return pd.read_csv(next(iter(subido)))
    except ImportError:
        raise FileNotFoundError(f"No encontré '{nombre}'. Cópielo en la misma carpeta de este notebook.")

---
# Parte 1 (guiada) · ¿El contrato marco reduce el precio de compra?

El área de abastecimiento compra el mismo insumo de dos formas: con **contrato marco** (proveedor negociado) o con **compra puntual** en el mercado. Gerencia quiere saber si el contrato marco realmente reduce el precio unitario antes de renovarlo por otro año.

Sean μ_C y μ_P los precios medios con contrato y con compra puntual:

$$H_0:\ \mu_C = \mu_P \qquad H_1:\ \mu_C \neq \mu_P$$

In [ ]:
compras = cargar_csv("IC07_precios_compras.csv")
display(compras.groupby("tipo_compra")["precio_unitario_miles"].describe().round(2))
sns.boxplot(data=compras, x="tipo_compra", y="precio_unitario_miles")
sns.stripplot(data=compras, x="tipo_compra", y="precio_unitario_miles", color="#333", alpha=.25, size=3)
plt.ylabel("precio unitario (miles de $)"); plt.title("Precio unitario según modalidad de compra"); plt.show()

### 1.1 Supuestos: forma de cada grupo y varianzas

La t de Student supone normalidad (o n grande) **y varianzas iguales**. La t de Welch solo necesita lo primero. Revisamos ambos supuestos.

In [ ]:
contrato = compras.loc[compras["tipo_compra"] == "Contrato marco", "precio_unitario_miles"]
puntual  = compras.loc[compras["tipo_compra"] == "Compra puntual", "precio_unitario_miles"]

for nombre, x in [("Contrato marco", contrato), ("Compra puntual", puntual)]:
    print(f"{nombre:15s} n = {len(x):3d}  asimetría = {x.skew():5.2f}  Shapiro p = {stats.shapiro(x).pvalue:.4f}")

F = puntual.var() / contrato.var()
p_F = 2 * stats.f.sf(F, len(puntual) - 1, len(contrato) - 1)
print(f"\nF = s²_puntual / s²_contrato = {F:.2f}   p = {p_F:.4g}")
print(f"Levene (mediana): p = {stats.levene(contrato, puntual, center='median').pvalue:.4g}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(contrato, line="s", ax=ax[0]); ax[0].set_title("Q-Q contrato marco")
sm.qqplot(puntual, line="s", ax=ax[1]); ax[1].set_title("Q-Q compra puntual")
plt.tight_layout(); plt.show()

### 1.2 Prueba de medias e intervalo de confianza

Como las varianzas son claramente distintas, se usa **Welch**. A modo de comparación se muestran también Student y Mann-Whitney.

In [ ]:
welch = stats.ttest_ind(contrato, puntual, equal_var=False)
student = stats.ttest_ind(contrato, puntual, equal_var=True)
mw = stats.mannwhitneyu(contrato, puntual)
print(f"Welch:        t = {welch.statistic:.3f}   p = {welch.pvalue:.3g}")
print(f"Student:      t = {student.statistic:.3f}   p = {student.pvalue:.3g}")
print(f"Mann-Whitney: U = {mw.statistic:.0f}   p = {mw.pvalue:.3g}")

ic = welch.confidence_interval(0.95)
dif = contrato.mean() - puntual.mean()
print(f"\nDiferencia de medias (contrato − puntual) = {dif:.2f} mil $   IC 95 %: [{ic.low:.2f} ; {ic.high:.2f}]")
print(f"Ahorro relativo: {-dif / puntual.mean():.1%} del precio de compra puntual")

**Pregunta 1.1.** ¿Por qué en este caso corresponde la t de Welch y no la de Student? ¿Cambia la conclusión si se usa Student?

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

**Pregunta 1.2.** Redacte la recomendación para gerencia: ¿conviene renovar el contrato marco? Use el intervalo de confianza para dimensionar el ahorro y mencione otro beneficio que se observa en los datos.

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

---
# Parte 2 (semi-guiada) · ¿Los proveedores entregan acero con la misma resistencia?

Calidad realizó 318 ensayos de tracción sobre bobinas de tres proveedores (Andes, Central y Pacífico). Se quiere saber si la resistencia media es la misma para los tres:

$$H_0:\ \mu_{Andes} = \mu_{Central} = \mu_{Pacífico} \qquad H_1: \text{al menos una media es diferente}$$

El ANOVA de un factor descompone la variabilidad total en la parte **entre grupos** y la parte **dentro de los grupos**: F = CM_entre / CM_dentro.

In [ ]:
proveedores = cargar_csv("IC05_resistencia_proveedores.csv")
display(proveedores.groupby("proveedor")["resistencia_traccion_mpa"].agg(["count", "mean", "std", "min", "max"]).round(2))
sns.boxplot(data=proveedores, x="proveedor", y="resistencia_traccion_mpa")
plt.ylabel("resistencia a la tracción (MPa)"); plt.show()

**2.1** Ajuste el modelo de ANOVA con `smf.ols` y muestre la tabla con `sm.stats.anova_lm`. Guarde el modelo en `modelo_prov`.

In [ ]:
# Pista: smf.ols("respuesta ~ C(factor)", data=...).fit()
# Escriba su código aquí
modelo_prov = ...

**2.2** Verifique los supuestos del ANOVA: normalidad de los **residuos** (Shapiro y gráfico Q-Q) e igualdad de varianzas entre proveedores (Levene).

In [ ]:
# Escriba su código aquí

**2.3** Como Levene rechaza, complemente con el **ANOVA de Welch** (`anova_oneway(..., use_var="unequal")`) y luego identifique qué pares difieren con Tukey (`pairwise_tukeyhsd`).

In [ ]:
# Escriba su código aquí

**Pregunta 2.1.** ¿Qué concluye sobre las medias de los tres proveedores? ¿Qué pares difieren y en cuánto?

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

**Pregunta 2.2.** Si la especificación exige una resistencia mínima de 450 MPa, ¿qué recomendaría al área de compras? Apoye su respuesta con la proporción de ensayos por debajo de 450 MPa en cada proveedor (calcúlela).

In [ ]:
# Escriba su código aquí

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

---
# Parte 3 (autónoma) · ¿El tiempo de reparación depende del tipo de equipo?

El área de mantenimiento registró 537 órdenes correctivas de cuatro familias de equipos (Empaque, Transportador, Prensa e Inyección). Quiere saber si el tiempo de reparación es el mismo en todas, para asignar técnicos y repuestos.

Realice un análisis completo: descripción por grupo, revisión de supuestos, la prueba adecuada y las comparaciones por pares. **Pista:** observe la forma de las distribuciones; los tiempos suelen ser asimétricos a la derecha. Considere dos caminos: Kruskal-Wallis sobre los tiempos o ANOVA sobre el **logaritmo** del tiempo.

In [ ]:
reparacion = cargar_csv("IC03_tiempos_reparacion.csv")
# Escriba su código aquí: descripción por grupo y gráficos

In [ ]:
# Escriba su código aquí: supuestos, prueba global y comparaciones por pares

**Pregunta 3.1.** ¿Qué prueba eligió y por qué? ¿Qué concluye? Explique cómo se interpretan las diferencias cuando se trabaja con el logaritmo del tiempo.

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._

**Pregunta 3.2.** ¿Qué decisión de gestión de mantenimiento sugieren estos resultados?

**✍️ Su respuesta:**

_Escriba aquí su análisis (doble clic para editar)._